In [1]:
import kagglehub
import os

path = kagglehub.dataset_download("andrewmvd/steam-reviews")
print("Path to dataset files:", path)
print(os.listdir(path))

c:\Users\ajdab\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Path to dataset files: C:\Users\ajdab\.cache\kagglehub\datasets\andrewmvd\steam-reviews\versions\3
['dataset.csv']


Importing the Data set. Our data set is a set of steam reviews.

It contains: 
Game ID, Game Name, Review Text, Review Score, and Review vote
Note: large data set, about 6.4M reviews and 2.16GB of space

Something to note in general: The actual data has about 5.26M Postive rows 1.16M Negative rows which roughly 82% positive so just take into account.

In [2]:
import pandas as pd

csv_path = os.path.join(path, "dataset.csv")   

cols = ["app_id", "app_name", "review_text", "review_score", "review_votes"]
df = pd.read_csv(csv_path, usecols=cols)

Firstly 6.4M rows is far to great to work with.

Want to take a subsample of rows and ensure its random as data is sorted by game.

In [21]:
sample_df = df.sample(n = 100000, random_state = 42) #make sure we use a seed for reproducibility


Next we will clean our data checking for duplicates, nulls, and empty strings, additionally wanted to look at review lenght.

In [22]:
cleaned_df = sample_df.copy()

# Replace missing reviews
if cleaned_df["review_text"].isnull().sum() > 0:
    cleaned_df["review_text"] = cleaned_df["review_text"].fillna("No review text")

# Remove empty reviews
cleaned_df = cleaned_df[cleaned_df["review_text"].str.strip() != ""]

# Remove duplicate reviews
cleaned_df = cleaned_df.drop_duplicates(subset=["review_text"])

# Calculate review length
cleaned_df["review_length"] = cleaned_df["review_text"].str.len()

download sklearn if not installed.

In [24]:
pip install -U scikit-learn

   ---------------------------------------- 0.0/8.4 MB ? eta -:--:--
   --------------------------- ------------ 5.8/8.4 MB 30.1 MB/s eta 0:00:01
   ---------------------------------------- 8.4/8.4 MB 24.7 MB/s  0:00:00
   ---------------------------------------- 0.0/37.4 MB ? eta -:--:--
   ------- -------------------------------- 7.3/37.4 MB 34.7 MB/s eta 0:00:01
   --------------- ------------------------ 14.7/37.4 MB 34.6 MB/s eta 0:00:01
   ----------------------- ---------------- 21.5/37.4 MB 33.3 MB/s eta 0:00:01
   ------------------------------ --------- 28.0/37.4 MB 32.7 MB/s eta 0:00:01
   ------------------------------------- -- 34.6/37.4 MB 32.5 MB/s eta 0:00:01
   ---------------------------------------- 37.4/37.4 MB 30.1 MB/s  0:00:01

   ------ --------------------------------- 1/6 [scipy]
   ------ --------------------------------- 1/6 [scipy]
   ------ --------------------------------- 1/6 [scipy]
   ------ --------------------------------- 1/6 [scipy]
   ------ -----


[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


import sklearn and set up our variables.

Additionally split our data sample into training and test data 80% and 20%.

In [39]:
import sklearn as sk
import sklearn.model_selection

x = cleaned_df["review_text"]
y = cleaned_df["review_score"]

#Split into training 80% and testing 20%
x_train, x_test, y_train, y_test = sk.model_selection.train_test_split(x, y, test_size = 0.2, random_state = 42, stratify = y)

Vectorize our review text, so we it can be read.

In [40]:
fit_transformer = sk.feature_extraction.text.TfidfVectorizer()


x_train_tfidf = fit_transformer.fit_transform(x_train)
x_test_tfidf = fit_transformer.transform(x_test)

In [41]:
print(x_train_tfidf.shape)
print(x_test_tfidf.shape)

(65820, 67071)
(16456, 67071)


Now check and make sure we are close to 82% 18% split. 

In [42]:
print(pd.Series(y_train).value_counts(normalize=True))
print(pd.Series(y_test).value_counts(normalize=True))

review_score
 1    0.82443
-1    0.17557
Name: proportion, dtype: float64
review_score
 1    0.824441
-1    0.175559
Name: proportion, dtype: float64


Get a basline model and predictions set up, will be used for scoring later.

In [44]:
import sklearn.dummy as dm#imports needed for a baseline model
import sklearn.metrics

baseline_model = dm.DummyClassifier(strategy = "most_frequent")
baseline_model.fit(x_train_tfidf, y_train) #learns the most frequent class from the training data

baseline_predictions =baseline_model.predict(x_test_tfidf)

Score the baseline prediction against the testing(y_test).

In [45]:
baseline_accuracy = sklearn.metrics.accuracy_score(y_test, baseline_predictions)
baseline_confusion_matrix = sklearn.metrics.confusion_matrix(y_test, baseline_predictions)
baseline_classification_report = sklearn.metrics.classification_report(y_test, baseline_predictions)

print("Baseline Model Accuracy:", baseline_accuracy)
print("Baseline Model Confusion Matrix:\n", baseline_confusion_matrix)
print("Baseline Model Classification Report:\n", baseline_classification_report)


Baseline Model Accuracy: 0.8244409333981526
Baseline Model Confusion Matrix:
 [[    0  2889]
 [    0 13567]]
Baseline Model Classification Report:
               precision    recall  f1-score   support

          -1       0.00      0.00      0.00      2889
           1       0.82      1.00      0.90     13567

    accuracy                           0.82     16456
   macro avg       0.41      0.50      0.45     16456
weighted avg       0.68      0.82      0.75     16456



c:\Users\ajdab\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\ajdab\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\ajdab\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(ave